# Automatización de Análisis de Documentos

## Objetivos de Aprendizaje

En este notebook, aprenderás a:

1. **Extraer información clave** de documentos con `CORTEX.COMPLETE()`
2. **Clasificar documentos** por tipo (contratos, DNIs, nóminas)
3. **Validar consistencia** entre documentos de un mismo expediente
4. **Automatizar la revisión** de solicitudes documentales
5. **Construir dashboard** de procesamiento documental

---

## Lo Que Construirás

Un **sistema de análisis documental automático**:
- Extracción de datos clave de documentos bancarios
- Clasificación automática por tipo de documento
- Validación cruzada de datos entre documentos
- Dashboard de estado de procesamiento

**Valor de Negocio**: Reducir tiempo de procesamiento documental de horas a minutos.

---

## Funcionalidades Clave

- `CORTEX.COMPLETE()` — Extracción y clasificación de documentos
- `CORTEX.SUMMARIZE()` — Resumen de documentos extensos
- `GENERATOR()` — Documentos sintéticos

¡Comencemos!

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS ANALISIS_DOCS_DB;
CREATE SCHEMA IF NOT EXISTS ANALISIS_DOCS_DB.PUBLIC;
USE SCHEMA ANALISIS_DOCS_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Generar Documentos Sintéticos

### Qué Vamos a Crear

- **500 expedientes** con 3-4 documentos cada uno
- **Tipos**: Contrato, DNI, Nómina, Extracto bancario, Escritura
- **Textos simulados** para extracción IA

In [ ]:
CREATE OR REPLACE TABLE DOCUMENTOS (
    doc_id VARCHAR(10) PRIMARY KEY,
    expediente_id VARCHAR(10),
    tipo_documento VARCHAR(30),
    contenido TEXT,
    fecha_recepcion DATE,
    estado VARCHAR(20)
);

INSERT INTO DOCUMENTOS
SELECT
    'DOC' || LPAD(SEQ4()::STRING, 6, '0'),
    'EXP' || LPAD(UNIFORM(1, 500, RANDOM())::STRING, 4, '0'),
    tipo,
    texto,
    DATEADD('day', -UNIFORM(0, 90, RANDOM()), CURRENT_DATE()),
    ARRAY_CONSTRUCT('Pendiente','En Revisión','Validado','Rechazado')[UNIFORM(0,3,RANDOM())]::VARCHAR
FROM (
    SELECT SEQ4() AS seq_id,
        CASE UNIFORM(1,5,RANDOM())
            WHEN 1 THEN 'Contrato'
            WHEN 2 THEN 'DNI'
            WHEN 3 THEN 'Nómina'
            WHEN 4 THEN 'Extracto Bancario'
            ELSE 'Escritura'
        END AS tipo,
        CASE UNIFORM(1,5,RANDOM())
            WHEN 1 THEN 'CONTRATO DE PRÉSTAMO PERSONAL. Entidad: Banco Nacional. Titular: María García López, DNI 12345678A. Importe: 25.000 EUR. Plazo: 60 meses. TAE: 6.5%. Cuota mensual: 489.32 EUR. Fecha firma: 15/03/2025.'
            WHEN 2 THEN 'DOCUMENTO NACIONAL DE IDENTIDAD. Nombre: Carlos Fernández Ruiz. DNI: 87654321B. Fecha nacimiento: 12/06/1985. Domicilio: Calle Mayor 15, 28001 Madrid. Vigencia: hasta 12/06/2030.'
            WHEN 3 THEN 'NÓMINA MENSUAL. Empresa: Tecnologías Avanzadas SL. Trabajador: Ana Martínez Soto. NIF empresa: B12345678. Salario bruto: 3.200 EUR. Deducciones: 640 EUR. Neto: 2.560 EUR. Período: Marzo 2025.'
            WHEN 4 THEN 'EXTRACTO BANCARIO. Cuenta: ES12 1234 5678 90 1234567890. Titular: Pedro López García. Período: 01/03/2025 - 31/03/2025. Saldo inicial: 15.234,56 EUR. Saldo final: 12.890,23 EUR. Movimientos: 45.'
            ELSE 'ESCRITURA DE COMPRAVENTA. Notario: D. José Pérez. Protocolo 1234/2025. Finca registral 12345. Comprador: Luis Sánchez. Vendedor: Inmobiliaria XYZ SL. Precio: 185.000 EUR. Cargas: Hipoteca 120.000 EUR.'
        END AS texto
    FROM TABLE(GENERATOR(ROWCOUNT => 1500))
);

SELECT tipo_documento, estado, COUNT(*) FROM DOCUMENTOS GROUP BY 1, 2 ORDER BY 1, 2;

---

## Paso 3: Extracción y Clasificación IA

### ¿Por Qué LLM para Documentos?

- Extracción automática de **datos clave** (nombres, importes, fechas)
- Clasificación sin reglas manuales
- Detección de **inconsistencias** entre documentos
- Escalable a miles de documentos

In [ ]:
-- Extraer datos clave con IA
CREATE OR REPLACE TABLE EXTRACCIONES AS
SELECT
    doc_id, expediente_id, tipo_documento, contenido,
    SNOWFLAKE.CORTEX.COMPLETE('mistral-large',
        'Extrae los datos clave de este documento bancario en formato JSON con campos: tipo, titular, importe, fecha, identificador. Si un campo no aplica, pon null.\n\nDocumento: ' || contenido || '\n\nResponde SOLO con el JSON.'
    ) AS datos_extraidos,
    SNOWFLAKE.CORTEX.COMPLETE('mistral-large',
        'Clasifica este documento en una categoría (Contrato/Identificación/Ingresos/Bancario/Inmobiliario) y asigna una confianza (Alta/Media/Baja). Responde: Categoría: X, Confianza: Y\n\nTexto: ' || contenido
    ) AS clasificacion_ia
FROM DOCUMENTOS
SAMPLE (100 ROWS);

SELECT doc_id, tipo_documento, datos_extraidos, clasificacion_ia
FROM EXTRACCIONES LIMIT 5;

---

## Paso 4: Resumen de Expedientes

Usamos `CORTEX.SUMMARIZE()` para generar resúmenes de expedientes completos.

In [ ]:
-- Resumen por expediente
CREATE OR REPLACE TABLE RESUMENES_EXPEDIENTE AS
WITH exp_docs AS (
    SELECT
        expediente_id,
        LISTAGG(tipo_documento || ': ' || LEFT(contenido, 100), ' | ') AS docs_concat,
        COUNT(*) AS num_docs
    FROM DOCUMENTOS
    GROUP BY 1
    HAVING COUNT(*) >= 3
)
SELECT
    expediente_id, num_docs,
    SNOWFLAKE.CORTEX.SUMMARIZE(docs_concat) AS resumen_expediente
FROM exp_docs
SAMPLE (30 ROWS);

SELECT * FROM RESUMENES_EXPEDIENTE LIMIT 5;

---

## Paso 5: Dashboard de Procesamiento Documental

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title('Automatización de Análisis de Documentos')
st.markdown('### Procesamiento Inteligente')

total = session.sql('SELECT COUNT(*) FROM DOCUMENTOS').collect()[0][0]
pendientes = session.sql("SELECT COUNT(*) FROM DOCUMENTOS WHERE estado='Pendiente'").collect()[0][0]
validados = session.sql("SELECT COUNT(*) FROM DOCUMENTOS WHERE estado='Validado'").collect()[0][0]

c1, c2, c3 = st.columns(3)
c1.metric('Total Documentos', f'{total:,}')
c2.metric('Pendientes', f'{pendientes:,}')
c3.metric('Validados', f'{validados:,}')

st.markdown('---')
st.subheader('Por Tipo de Documento')
df = session.sql('SELECT tipo_documento, COUNT(*) AS n FROM DOCUMENTOS GROUP BY 1 ORDER BY 2 DESC').to_pandas()
st.bar_chart(df.set_index('TIPO_DOCUMENTO'))

st.markdown('---')
st.subheader('Extracciones IA')
df_ext = session.sql('SELECT doc_id, tipo_documento, datos_extraidos, clasificacion_ia FROM EXTRACCIONES LIMIT 10').to_pandas()
st.dataframe(df_ext)

---

## Paso 6: Limpieza (Opcional)

In [ ]:
-- Descomentar para limpiar

-- DROP DATABASE IF EXISTS ANALISIS_DOCS_DB;